# Converter Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from power_electronics.converters.dc_dc import BuckConverter, BoostConverter, BuckBoostConverter, CukConverter, SepicConverter, FlybackConverter, ForwardConverter, DualActiveBridgeConverter
from power_electronics.converters.dc_ac import SinglePhaseHBridgeVSI, ThreePhaseVSI, NPCInverter

In [ ]:
loads=np.linspace(5.0,80.0,18); models={'buck':BuckConverter,'boost':BoostConverter,'buck_boost':BuckBoostConverter,'cuk':CukConverter,'sepic':SepicConverter,'flyback':FlybackConverter,'forward':ForwardConverter,'dab':DualActiveBridgeConverter}
plt.figure(figsize=(8,4))
for name,cls in models.items():
    eff=[]
    for R in loads:
        p={'Vin':48,'duty_cycle':0.4,'frequency':20000,'L':1e-3,'C':220e-6,'R_load':float(R),'R_L':0.05,'n':2.0,'phase_shift':0.25}
        eff.append(cls(p).simulate(0.03,4000).metrics['efficiency_%'])
    plt.plot(loads,eff,label=name)
plt.xlabel('R_load (ohm)'); plt.ylabel('Efficiency (%)'); plt.grid(True,alpha=0.3); plt.legend(ncol=2,fontsize=8)

In [ ]:
freq=np.linspace(5000,60000,16)
for cls,name in [(BuckConverter,'buck'),(BoostConverter,'boost'),(BuckBoostConverter,'buck_boost')]:
    rip=[]
    for f in freq:
        r=cls({'Vin':48,'duty_cycle':0.4,'frequency':float(f),'L':1e-3,'C':220e-6,'R_load':15,'R_L':0.05}).simulate(0.03,3000)
        rip.append(r.metrics['Vout_ripple_%'])
    plt.plot(freq,rip,label=name)
plt.xlabel('f_sw (Hz)'); plt.ylabel('Ripple (%)'); plt.grid(True,alpha=0.3); plt.legend()

In [ ]:
mi=np.linspace(0.2,1.0,10)
for cls,name in [(SinglePhaseHBridgeVSI,'hbridge'),(ThreePhaseVSI,'three_phase_vsi'),(NPCInverter,'npc')]:
    thd=[]
    for m in mi:
        r=cls({'Vdc':300,'f_output':50,'f_carrier':5000,'modulation_index':float(m),'R_load':20,'L_load':0.01,'C_load':0.0,'control_mode':'SPWM'}).simulate(0.08,3500)
        thd.append(r.metrics['THD_%'])
    plt.plot(mi,thd,label=name)
plt.xlabel('Modulation index'); plt.ylabel('THD (%)'); plt.grid(True,alpha=0.3); plt.legend()

In [ ]:
labels=['efficiency','THD','ripple','cost','complexity']
scores={'buck':[90,60,65,85,70],'boost':[88,58,62,82,68],'three_phase_vsi':[92,72,75,60,55],'npc':[95,80,82,45,40]}
ang=np.linspace(0,2*np.pi,len(labels),endpoint=False).tolist(); ang+=ang[:1]
fig=plt.figure(figsize=(6,6)); ax=plt.subplot(111,polar=True)
for n,v in scores.items():
    vv=v+v[:1]; ax.plot(ang,vv,label=n); ax.fill(ang,vv,alpha=0.1)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels); ax.legend(loc='upper right',bbox_to_anchor=(1.25,1.1))